In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


# Playground Series S6E6 Kaggle Notebook Cells

## 1. Import + Config

In [2]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


class CFG:
    competition = "playground-series-s6e6"
    exp_id = "exp_008_add-colour-curvature"
    parent_exp_id = "exp_007"
    note = "Add colour curvature"

    data_dir_candidates = [
        Path("/kaggle/input/playground-series-s6e6"),
        Path("/kaggle/input/competitions/playground-series-s6e6"),
    ]
    output_path = Path("submission.csv")

    seed = 42
    n_splits = 5
    models = ["lgbm"]
    model_weights = None  # None means simple average.

    num_boost_round = 5000
    early_stopping_rounds = 100
    verbose_eval = 100

    lgbm_params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "learning_rate": 0.03,
        "num_leaves": 64,
        "max_depth": -1,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "feature_fraction": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_child_samples": 30,
        "seed": seed,
        "n_jobs": -1,
        "verbosity": -1,
        "device": "gpu",
    }

    xgb_params = {
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "eta": 0.03,
        "max_depth": 7,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "lambda": 1.0,
        "alpha": 0.1,
        "seed": seed,
        "tree_method": "hist",
        "device": "cuda",
    }

    cat_params = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": num_boost_round,
        "learning_rate": 0.03,
        "depth": 7,
        "l2_leaf_reg": 3.0,
        "random_seed": seed,
        "task_type": "GPU",
        "devices": "0",
        "verbose": verbose_eval,
        "allow_writing_files": False,
    }


def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(CFG.seed)

## 2. Data Loading

In [3]:
def find_data_dir(candidates):
    for data_dir in candidates:
        if (data_dir / "train.csv").exists():
            return data_dir
    raise FileNotFoundError("Could not find train.csv. Check Kaggle input settings.")


CFG.data_dir = find_data_dir(CFG.data_dir_candidates)

train = pd.read_csv(CFG.data_dir / "train.csv")
test = pd.read_csv(CFG.data_dir / "test.csv")
sample_submission = pd.read_csv(CFG.data_dir / "sample_submission.csv")

id_col = "id" if "id" in train.columns else sample_submission.columns[0]
target_col = [c for c in sample_submission.columns if c != id_col][0]

print("data_dir:", CFG.data_dir)
print("train:", train.shape)
print("test :", test.shape)
print("target:", target_col)
print(train.head())

data_dir: /kaggle/input/competitions/playground-series-s6e6
train: (577347, 12)
test : (247435, 11)
target: class
   id       alpha      delta          u          g          r          i  \
0   0  147.734256  16.959273  25.472123  21.895559  20.357926  19.257113   
1   1  127.988677  32.346716  20.778509  19.087062  17.587208  17.226067   
2   2  179.792648  35.344843  21.035203  21.079128  21.171840  20.582629   
3   3  225.818295  48.569421  23.305056  21.050736  19.017754  18.365658   
4   4  141.836135  19.342852  21.703158  19.471680  18.234449  17.899447   

           z  redshift spectral_type galaxy_population   class  
0  18.621057  0.408982             M      Red_Sequence  GALAXY  
1  16.786433  0.157976             M      Red_Sequence  GALAXY  
2  20.557366  2.823770           O/B        Blue_Cloud     QSO  
3  17.914952  0.536099             M      Red_Sequence  GALAXY  
4  17.616185  0.555761             M      Red_Sequence  GALAXY  


## 3. Feature Generation Hooks

In [4]:
def add_basic_features(df):
    # Future extraction point: src/features.py
    df = df.copy()

    feature_cols = [c for c in df.columns if c not in [id_col, target_col]]
    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

    if len(num_cols) > 0:
        df["num_missing_count"] = df[num_cols].isna().sum(axis=1)
        df["num_zero_count"] = (df[num_cols] == 0).sum(axis=1)

    # Vectorisation
    if "alpha" in df.columns:
        alpha_rad = np.radians(df["alpha"])
        df["alpha_sin"] = np.sin(alpha_rad)
        df["alpha_cos"] = np.cos(alpha_rad)

    if "delta" in df.columns:
        delta_rad = np.radians(df["delta"])
        df["delta_sin"] = np.sin(delta_rad)
        df["delta_cos"] = np.cos(delta_rad)

    # Colour Indices
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    # Spectral Slope
    df["u_z"] = df["u"] - df["z"]
    df["u_i"] = df["u"] - df["i"]
    df["g_z"] = df["g"] - df["z"]

    # Colour Curvature
    df["colour_curvature_ug_gr"] = df["u_g"] - df["g_r"]
    df["colour_curvature_gr_ri"] = df["g_r"] - df["r_i"]
    df["colour_curvature_ri_iz"] = df["r_i"] - df["i_z"]
    
    return df


def prepare_features(train_df, test_df):
    train_df = add_basic_features(train_df)
    test_df = add_basic_features(test_df)

    features = [c for c in train_df.columns if c not in [id_col, target_col]]
    all_df = pd.concat([train_df[features], test_df[features]], axis=0, ignore_index=True)

    cat_cols = [c for c in features if all_df[c].dtype == "object" or str(all_df[c].dtype) == "category"]
    num_cols = [c for c in features if c not in cat_cols]

    for c in num_cols:
        median = all_df[c].median()
        train_df[c] = train_df[c].fillna(median)
        test_df[c] = test_df[c].fillna(median)

    for c in cat_cols:
        train_df[c] = train_df[c].astype("string").fillna("__MISSING__")
        test_df[c] = test_df[c].astype("string").fillna("__MISSING__")

        cats = pd.Index(pd.concat([train_df[c], test_df[c]], axis=0).unique())
        train_df[c] = pd.Categorical(train_df[c], categories=cats)
        test_df[c] = pd.Categorical(test_df[c], categories=cats)

    return train_df, test_df, features, cat_cols


train_fe, test_fe, features, cat_cols = prepare_features(train, test)

print("n_features:", len(features))
print("categorical:", cat_cols)

n_features: 26
categorical: ['spectral_type', 'galaxy_population']


## 4. Validation

In [5]:
def encode_target(values):
    classes = pd.Index(pd.unique(values))
    class_to_id = {label: i for i, label in enumerate(classes)}
    encoded = pd.Series(values).map(class_to_id).to_numpy()
    return encoded, classes


def make_stratified_folds(y, n_splits, seed):
    rng = np.random.default_rng(seed)
    fold_indices = [[] for _ in range(n_splits)]

    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        for fold, part in enumerate(np.array_split(cls_idx, n_splits)):
            fold_indices[fold].extend(part.tolist())

    all_idx = np.arange(len(y))
    folds = []
    for valid_idx in fold_indices:
        valid_idx = np.array(sorted(valid_idx))
        train_idx = np.setdiff1d(all_idx, valid_idx)
        folds.append((train_idx, valid_idx))
    return folds


def accuracy(y_true, proba):
    return float((y_true == proba.argmax(axis=1)).mean())


def multiclass_log_loss(y_true, proba):
    proba = np.clip(proba, 1e-15, 1 - 1e-15)
    return float(-np.mean(np.log(proba[np.arange(len(y_true)), y_true])))


y, classes = encode_target(train_fe[target_col])
n_classes = len(classes)
folds = make_stratified_folds(y, CFG.n_splits, CFG.seed)

print("classes:", list(classes))
print("folds:", len(folds))

classes: ['GALAXY', 'QSO', 'STAR']
folds: 5


## 5. Training

In [6]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool


def encode_categoricals(df):
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].cat.codes.astype("int32")
    return out


def train_lgbm_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Future extraction point: src/train.py
    params = CFG.lgbm_params.copy()
    params["num_class"] = n_classes

    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
    valid_data = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_cols, free_raw_data=False)

    model = lgb.train(
        params,
        train_data,
        num_boost_round=CFG.num_boost_round,
        valid_sets=[valid_data],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(CFG.early_stopping_rounds, verbose=False),
            lgb.log_evaluation(CFG.verbose_eval),
        ],
    )
    valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)
    test_pred = model.predict(X_test, num_iteration=model.best_iteration)
    return model, valid_pred, test_pred, model.best_iteration


def train_xgb_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Future extraction point: src/train.py
    params = CFG.xgb_params.copy()
    params["num_class"] = n_classes

    X_train_xgb = encode_categoricals(X_train)
    X_valid_xgb = encode_categoricals(X_valid)
    X_test_xgb = encode_categoricals(X_test)

    dtrain = xgb.DMatrix(X_train_xgb, label=y_train)
    dvalid = xgb.DMatrix(X_valid_xgb, label=y_valid)
    dtest = xgb.DMatrix(X_test_xgb)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=CFG.num_boost_round,
        evals=[(dvalid, "valid")],
        early_stopping_rounds=CFG.early_stopping_rounds,
        verbose_eval=CFG.verbose_eval,
    )
    valid_pred = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    test_pred = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
    return model, valid_pred, test_pred, model.best_iteration


def to_catboost_frame(df):
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].astype(str)
    return out


def train_cat_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Future extraction point: src/train.py
    cat_features = [X_train.columns.get_loc(c) for c in cat_cols]

    X_train_cat = to_catboost_frame(X_train)
    X_valid_cat = to_catboost_frame(X_valid)
    X_test_cat = to_catboost_frame(X_test)

    train_pool = Pool(X_train_cat, y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid_cat, y_valid, cat_features=cat_features)
    test_pool = Pool(X_test_cat, cat_features=cat_features)

    model = CatBoostClassifier(**CFG.cat_params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=CFG.early_stopping_rounds,
        use_best_model=True,
    )
    valid_pred = model.predict_proba(valid_pool)
    test_pred = model.predict_proba(test_pool)
    return model, valid_pred, test_pred, model.get_best_iteration()


def train_model(model_name):
    oof = np.zeros((len(train_fe), n_classes), dtype=np.float32)
    test_pred = np.zeros((len(test_fe), n_classes), dtype=np.float32)
    fold_logs = []

    print(f"\n===== {model_name} =====")
    for fold, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr = train_fe.iloc[tr_idx][features]
        y_tr = y[tr_idx]
        X_va = train_fe.iloc[va_idx][features]
        y_va = y[va_idx]
        X_te = test_fe[features]

        if model_name == "lgbm":
            model, valid_pred, test_fold_pred, best_iteration = train_lgbm_fold(X_tr, y_tr, X_va, y_va, X_te)
        elif model_name == "xgb":
            model, valid_pred, test_fold_pred, best_iteration = train_xgb_fold(X_tr, y_tr, X_va, y_va, X_te)
        elif model_name == "cat":
            model, valid_pred, test_fold_pred, best_iteration = train_cat_fold(X_tr, y_tr, X_va, y_va, X_te)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        oof[va_idx] = valid_pred
        test_pred += test_fold_pred / CFG.n_splits

        fold_acc = accuracy(y_va, valid_pred)
        fold_ll = multiclass_log_loss(y_va, valid_pred)
        fold_logs.append({
            "fold": fold,
            "accuracy": fold_acc,
            "logloss": fold_ll,
            "best_iteration": int(best_iteration) if best_iteration is not None else None,
        })
        print(f"{model_name} fold {fold}: accuracy={fold_acc:.6f}, logloss={fold_ll:.6f}, best_iteration={best_iteration}")

        del model, X_tr, X_va, X_te
        gc.collect()

    cv_acc = accuracy(y, oof)
    cv_ll = multiclass_log_loss(y, oof)
    print(f"{model_name} CV: accuracy={cv_acc:.6f}, logloss={cv_ll:.6f}")

    return {
        "name": model_name,
        "oof": oof,
        "test_pred": test_pred,
        "cv_accuracy": cv_acc,
        "cv_logloss": cv_ll,
        "folds": fold_logs,
    }


model_results = []
for model_name in CFG.models:
    model_results.append(train_model(model_name))


===== lgbm =====


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[100]	valid's multi_logloss: 0.116915
[200]	valid's multi_logloss: 0.0933971
[300]	valid's multi_logloss: 0.0893143
[400]	valid's multi_logloss: 0.0878343
[500]	valid's multi_logloss: 0.0870548
[600]	valid's multi_logloss: 0.0866217
[700]	valid's multi_logloss: 0.0863391
[800]	valid's multi_logloss: 0.0861527
[900]	valid's multi_logloss: 0.086038
[1000]	valid's multi_logloss: 0.0859229
[1100]	valid's multi_logloss: 0.0858605
[1200]	valid's multi_logloss: 0.0858093
[1300]	valid's multi_logloss: 0.0857931
lgbm fold 1: accuracy=0.969213, logloss=0.085774, best_iteration=1240
[100]	valid's multi_logloss: 0.116375
[200]	valid's multi_logloss: 0.0930965
[300]	valid's multi_logloss: 0.089446
[400]	valid's multi_logloss: 0.0881984
[500]	valid's multi_logloss: 0.0874534
[600]	valid's multi_logloss: 0.0869435
[700]	valid's multi_logloss: 0.0866836
[800]	valid's multi_logloss: 0.086371
[900]	valid's multi_logloss: 0.0862803
[1000]	valid's multi_logloss: 0.0862219
[1100]	valid's multi_logloss: 0.0

## 6. Inference

In [7]:
if CFG.model_weights is None:
    weights = np.ones(len(model_results), dtype=np.float32) / len(model_results)
else:
    weights = np.array(CFG.model_weights, dtype=np.float32)
    weights = weights / weights.sum()

model_names = [result["name"] for result in model_results]
oof_proba = sum(w * result["oof"] for w, result in zip(weights, model_results))
test_proba = sum(w * result["test_pred"] for w, result in zip(weights, model_results))

cv_acc = accuracy(y, oof_proba)
cv_ll = multiclass_log_loss(y, oof_proba)

test_pred = classes.to_numpy()[test_proba.argmax(axis=1)]
pred_distribution = pd.Series(test_pred).value_counts(normalize=True).sort_index()

print("models:", model_names)
print("weights:", dict(zip(model_names, weights)))
print(f"ensemble CV: accuracy={cv_acc:.6f}, logloss={cv_ll:.6f}")
print(pred_distribution)

models: ['lgbm']
weights: {'lgbm': np.float32(1.0)}
ensemble CV: accuracy=0.968887, logloss=0.087030
GALAXY    0.655101
QSO       0.202199
STAR      0.142700
Name: proportion, dtype: float64


## 7. Submission

In [8]:
submission = sample_submission.copy()
submission[target_col] = test_pred
submission.to_csv(CFG.output_path, index=False)

experiment_log = {
    "competition": CFG.competition,
    "exp_id": CFG.exp_id,
    "parent_exp_id": CFG.parent_exp_id,
    "note": CFG.note,
    "seed": CFG.seed,
    "n_splits": CFG.n_splits,
    "models": model_names,
    "model_weights": dict(zip(model_names, [float(w) for w in weights])),
    "model_scores": [
        {
            "name": result["name"],
            "cv_accuracy": float(result["cv_accuracy"]),
            "cv_logloss": float(result["cv_logloss"]),
            "folds": result["folds"],
        }
        for result in model_results
    ],
    "ensemble_cv_accuracy": float(cv_acc),
    "ensemble_cv_logloss": float(cv_ll),
    "n_features": len(features),
    "categorical_features": cat_cols,
    "gpu_settings": {
        "lgbm_device": CFG.lgbm_params.get("device"),
        "xgb_device": CFG.xgb_params.get("device"),
        "cat_task_type": CFG.cat_params.get("task_type"),
    },
    "output_path": str(CFG.output_path),
}

print(submission.head())
print("saved:", CFG.output_path)
experiment_log

       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
saved: submission.csv


{'competition': 'playground-series-s6e6',
 'exp_id': 'exp_008_add-colour-curvature',
 'parent_exp_id': 'exp_007',
 'note': 'Add colour curvature',
 'seed': 42,
 'n_splits': 5,
 'models': ['lgbm'],
 'model_weights': {'lgbm': 1.0},
 'model_scores': [{'name': 'lgbm',
   'cv_accuracy': 0.9688869951692829,
   'cv_logloss': 0.08702972531318665,
   'folds': [{'fold': 1,
     'accuracy': 0.9692127825409197,
     'logloss': 0.08577400343856371,
     'best_iteration': 1240},
    {'fold': 2,
     'accuracy': 0.9696717762189313,
     'logloss': 0.08611033652973867,
     'best_iteration': 1257},
    {'fold': 3,
     'accuracy': 0.9681648913137612,
     'logloss': 0.08686787275978132,
     'best_iteration': 1226},
    {'fold': 4,
     'accuracy': 0.9681646156111164,
     'logloss': 0.09002718617560339,
     'best_iteration': 870},
    {'fold': 5,
     'accuracy': 0.9692209096892646,
     'logloss': 0.08636921508975917,
     'best_iteration': 1282}]}],
 'ensemble_cv_accuracy': 0.9688869951692829,
 'e